# Combo 4 - Chakra teacher-student distillation

This Kaggle notebook trains a compact MobileNetV3 student from the multi-scale Chakra ViT teacher. Upload the teacher checkpoint as a Kaggle Dataset, or leave the checkpoint path empty to use the ImageNet-initialized teacher. The student is the deployment candidate for live video; the ViT remains the accuracy reference.

In [ ]:
!git clone -q https://github.com/varshasrao58/chakramodel.git /kaggle/working/chakramodel
%cd /kaggle/working/chakramodel
!pip install -q timm==0.9.16 opencv-python-headless
import os, glob, shutil, subprocess
assert os.path.exists('/kaggle/input'), 'Run this notebook on Kaggle with GPU enabled'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode())

In [ ]:
from pathlib import Path
import os, glob, shutil

def find_pair_dirs(root):
    image_candidates, mask_candidates = [], []
    for path in Path(root).rglob('*'):
        if path.is_dir() and path.name.lower() in {'images', 'image', 'frames'}: image_candidates.append(path)
        if path.is_dir() and path.name.lower() in {'masks', 'mask', 'ground_truth'}: mask_candidates.append(path)
    for images in image_candidates:
        for masks in mask_candidates:
            if len(list(images.glob('*'))) and len(list(masks.glob('*'))): return images, masks
    return None, None
images, masks = find_pair_dirs('/kaggle/input')
if images is None:
    raise FileNotFoundError('Attach a Kaggle Dataset containing paired images/ and masks/ directories')
target_images = Path('/kaggle/working/chakramodel/data/kvasir-seg/images')
target_masks = Path('/kaggle/working/chakramodel/data/kvasir-seg/masks')
target_images.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(images, target_images, dirs_exist_ok=True)
shutil.copytree(masks, target_masks, dirs_exist_ok=True)
teacher_matches = glob.glob('/kaggle/input/**/*.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pth', recursive=True)
teacher = teacher_matches[0] if teacher_matches else ''
print(f'images={len(list(target_images.glob("*")))} masks={len(list(target_masks.glob("*")))} teacher={teacher or "ImageNet initialization"}')

In [ ]:
import subprocess, sys
command = [sys.executable, 'src/chakra_transformer/train_distill.py', '--epochs', '30', '--batch-size', '8', '--grad-accum', '2', '--workers', '2']
if teacher: command += ['--teacher-checkpoint', teacher]
subprocess.run(command, check=True)

In [ ]:
import torch, time
from src.chakra_transformer.student_segmenter import ChakraLiveStudent
student = ChakraLiveStudent().cuda().eval()
student.load_state_dict(torch.load('weights/combo4_student_best.pth', map_location='cuda'))
x = torch.randn(1, 3, 352, 352, device='cuda')
for _ in range(20): student(x)
torch.cuda.synchronize(); start = time.perf_counter()
for _ in range(100): student(x)
torch.cuda.synchronize(); elapsed = time.perf_counter() - start
print(f'parameters={sum(p.numel() for p in student.parameters()):,}; FPS={100 / elapsed:.2f}; latency_ms={elapsed * 10:.2f}')
shutil.copy('weights/combo4_student_best.pth', '/kaggle/working/combo4_student_best.pth')